# 나카무라 뼈대(포즈) 추출 — Colab 무료 GPU

순서대로 실행하거나, 메뉴 **런타임 → 모두 실행**을 누른다.

- 시작 전: **런타임 → 런타임 유형 변경 → T4 GPU** 선택
- 2번 칸에서 Google Drive 연결을 허용한다.
- 끊겨도 다시 '모두 실행'하면 끝난 영상은 건너뛰고 이어서 한다.
- 결과: 내 드라이브 `AI-sfa/02_분석자료/포즈/nakamura/` 와 `nakamura_pose.zip`

In [ ]:
# 1. 준비(1~2분)
!pip -q install ultralytics
import torch; print('GPU:', torch.cuda.is_available() and torch.cuda.get_device_name(0))

In [ ]:
# 2. 내 드라이브 연결(허용 창이 뜨면 계정 선택 → 허용)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 설정
import os, json, gzip, time, shutil
SRC = '/content/drive/MyDrive/AI-sfa/01_원본영상/_Gemini용_저용량/yutaka_nakamura'
OUT = '/content/drive/MyDrive/AI-sfa/02_분석자료/포즈/nakamura'
os.makedirs(OUT, exist_ok=True)
IDS = ["108301", "109047", "110193", "110822", "110865", "111284", "115592", "123722", "12557", "130581", "130604", "13542", "13575", "13576", "13577", "13578", "13579", "141366", "150800", "152142", "152143", "152144", "157400", "159142", "159418", "162763", "162980", "165486", "166565", "167641", "167784", "167785", "174308", "174312", "174315", "174333", "174685", "181174", "182469", "182575", "182678", "182793", "182904", "196315", "196873", "196994", "202164", "202176", "202388", "203045", "203287", "203289", "203290", "203291", "203292", "203293", "203294", "203326", "203327", "203328", "205025", "205406", "205408", "205409", "208545", "209789", "209790", "210085", "210213", "210214", "210215", "210218", "210219", "210343", "212631", "213601", "216504", "216509", "217673", "218099", "218752", "218838", "220509", "220604", "220613", "220930", "220969", "222547", "224077", "226853", "227508", "229682", "229832", "229975", "230555", "231052", "232394", "232580", "233386", "234710", "236056", "236211", "236213", "236215", "239055", "239063", "239068", "239082", "239750", "239776", "24634", "247829", "247995", "250125", "250758", "250896", "252543", "253949", "256092", "258247", "258994", "263191", "266832", "269366", "269863", "271632", "271634", "271642", "271643", "271644", "272649", "272715", "273325", "273677", "280210", "280456", "281184", "281196", "281209", "281251", "281565", "281605", "281606", "285111", "290849", "290894", "290900", "292678", "294864", "295424", "295427", "299654", "301777", "302394", "306329", "306713", "309107", "309311", "309515", "3626", "374", "38525", "40686", "40687", "40688", "41027", "41061", "42687", "42723", "43901", "43903", "47907", "506", "69067", "69068", "69069", "74684", "75672", "76453", "78", "79", "80", "81", "82", "8645", "9370"]
have = [i for i in IDS if os.path.exists(f'{SRC}/{i}.mp4')]
print('대상', len(IDS), '드라이브에 있음', len(have), '이미 끝남', sum(os.path.exists(f'{OUT}/{i}.jsonl.gz') for i in IDS))

In [ ]:
# 4. 포즈 추출(모든 프레임, 약 30~50분). 영상마다 한 줄씩 진행 상황이 찍힌다.
from ultralytics import YOLO
model = YOLO('yolo11m-pose.pt')
t0 = time.time()
for k, vid in enumerate(have):
    dst = f'{OUT}/{vid}.jsonl.gz'
    if os.path.exists(dst):
        continue
    tmp = f'/content/{vid}.jsonl.gz'
    n = det = 0
    with gzip.open(tmp, 'wt', encoding='utf-8') as f:
        for i, res in enumerate(model.predict(f'{SRC}/{vid}.mp4', stream=True, verbose=False, half=True, imgsz=640, conf=0.25)):
            people = []
            if res.keypoints is not None and len(res.boxes):
                kp = res.keypoints.data.cpu().numpy().round(1).tolist()
                bx = res.boxes.xyxy.cpu().numpy().round(1).tolist()
                cf = res.boxes.conf.cpu().numpy().round(3).tolist()
                people = [{'box': bx[j], 'conf': cf[j], 'kpts': kp[j]} for j in range(len(bx))]
            f.write(json.dumps({'f': i, 'p': people}) + '\n')
            n += 1; det += bool(people)
    shutil.move(tmp, dst)  # 로컬에 다 쓴 뒤 드라이브로 옮김(중간에 끊겨도 반쪽 파일이 남지 않음)
    print(k + 1, len(have), vid, '프레임', n, '사람 검출', det, f'{time.time()-t0:.0f}초', flush=True)
print('끝')

In [ ]:
# 5. 결과를 zip 하나로 묶기(이 파일을 Claude 채팅에 첨부)
import shutil
zp = shutil.make_archive('/content/drive/MyDrive/AI-sfa/02_분석자료/포즈/nakamura_pose', 'zip', OUT)
print(zp, round(os.path.getsize(zp)/1e6, 1), 'MB', '파일 수', len([x for x in os.listdir(OUT) if x.endswith('.gz')]))